In [1]:
# Ensure repo root is on sys.path so package imports like `Winrate_Prediction.src.*` work
import sys
import os
from pathlib import Path
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Added to sys.path:', repo_root)

Added to sys.path: c:\Users\Student\OneDrive - Birmingham City University\2nd Year\MLOPS\Assignment\Winrate_Prediction\notebooks


# Feature Exploration and Model Summary
This notebook loads the prepared features, shows basic distributions, and visualizes model feature importances if a trained model exists.

## Prediction target
We predict whether the blue side wins the match. The target column is `blue_side_win` (binary).
The following sections show the target distribution and example feature / champion-level summaries to help reviewers understand what we're predicting.

In [2]:
# Display per-role summary CSVs (if present) for reviewers
import glob
from IPython.display import display
per_role_files = sorted(glob.glob(str(Path('Winrate_Prediction')/ 'analysis_outputs' / '*_per_champion_stats.csv')))
dfs = []
for f in per_role_files:
    role = Path(f).name.split('_')[0]
    try:
        df_role = pd.read_csv(f)
    except Exception as e:
        print('Could not read', f, '->', e)
        continue
    name_col = next((c for c in df_role.columns if 'name' in c.lower() or 'champ' in c.lower()), None)
    games_col = next((c for c in df_role.columns if any(k in c.lower() for k in ['count','games','plays','games_played','play_count'])), None)
    win_col = next((c for c in df_role.columns if 'win' in c.lower()), None)
    if name_col is None:
        continue
    cols = [name_col]
    if games_col: cols.append(games_col)
    if win_col: cols.append(win_col)
    df2 = df_role[cols].copy()
    rename_map = {name_col: 'champion'}
    if games_col: rename_map[games_col] = 'games'
    if win_col: rename_map[win_col] = 'win_rate'
    df2 = df2.rename(columns=rename_map)
    df2['role'] = role
    dfs.append(df2)
if dfs:
    combined = pd.concat(dfs, ignore_index=True, sort=False)
    combined['champion'] = combined['champion'].astype(str)
    display(combined.sort_values(by=['role'] + ([ 'games' ] if 'games' in combined.columns else []), ascending=[True]+([False] if 'games' in combined.columns else [])).reset_index(drop=True).head(200))
else:
    print('No per-role summary CSVs found at Winrate_Prediction/analysis_outputs')

No per-role summary CSVs found at Winrate_Prediction/analysis_outputs


In [3]:
# Top champions per role (plots) -- executed notebook version
import numpy as np
if 'combined' in globals() and not combined.empty:
    roles = sorted(combined['role'].unique())
    for role in roles:
        df_r = combined[combined['role']==role].copy()
        if df_r.empty:
            continue
        if 'games' in df_r.columns:
            df_r = df_r.sort_values('games', ascending=False).head(8)
            xcol = 'games'
        else:
            df_r = df_r.sort_values('win_rate', ascending=False).head(8)
            xcol = 'win_rate'
        plt.figure(figsize=(8,4))
        sns.barplot(x=xcol, y='champion', data=df_r, palette='viridis')
        plt.title(f'Top champs for role {role}')
        if 'win_rate' in df_r.columns:
            for i, row in enumerate(df_r.itertuples()):
                val = getattr(row, xcol, np.nan)
                if xcol == 'games':
                    ann = f" win:{getattr(row, 'win_rate', np.nan):.2f}" if 'win_rate' in df_r.columns else ''
                    plt.text(val + max(1, val*0.01), i, ann, va='center')
                else:
                    ann = f" games:{int(getattr(row, 'games', 0))}" if 'games' in df_r.columns else ''
                    plt.text(val + 0.01, i, ann, va='center')
        plt.tight_layout()
        plt.show()
else:
    print('No per-role summary table available; skipping top-champion plots')

No per-role summary table available; skipping top-champion plots


In [4]:
# Compact summary: avg win rate and total games per role
if 'combined' in globals() and not combined.empty:
    if 'games' in combined.columns and 'win_rate' in combined.columns:
        rs = combined.groupby('role').apply(lambda g: pd.Series({'avg_win_rate': (g['win_rate']*g['games']).sum()/g['games'].sum() if g['games'].sum()>0 else g['win_rate'].mean(), 'total_games': int(g['games'].sum())})).reset_index()
    elif 'win_rate' in combined.columns:
        rs = combined.groupby('role').agg(avg_win_rate=('win_rate','mean'), total_games=('win_rate','size')).reset_index()
    else:
        rs = pd.DataFrame({'role':[], 'avg_win_rate':[], 'total_games':[]})
    if not rs.empty:
        rs = rs.sort_values('avg_win_rate', ascending=False).reset_index(drop=True)
        fig, axes = plt.subplots(1,2, figsize=(12,5), gridspec_kw={'width_ratios':[1,1]})
        sns.barplot(x='avg_win_rate', y='role', data=rs, ax=axes[0], palette='coolwarm')
        axes[0].set_xlim(0,1)
        axes[0].set_title('Avg win rate by role')
        sns.barplot(x='total_games', y='role', data=rs, ax=axes[1], palette='Greens')
        axes[1].set_xscale('log')
        axes[1].set_title('Total games by role (log scale)')
        plt.tight_layout()
        plt.show()
    else:
        print('Role summary could not be computed (missing columns).')
else:
    print('No per-role summary table available; skipping compact summary')

No per-role summary table available; skipping compact summary


In [5]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
FEATURES_PATH = Path('../data/processed/features.parquet')
# Model path (repo root models directory)
MODEL_PATH = Path('../../models/xgb_patch_model.pkl')
if not MODEL_PATH.exists():
    print('Model not found at', MODEL_PATH, '; expected at repo root models/xgb_patch_model.pkl')
else:
    print('Using model at', MODEL_PATH)
if not FEATURES_PATH.exists():
    print('Features file not found at', FEATURES_PATH, '- skipping loading and visualizations.')
    df = pd.DataFrame()
else:
    df = pd.read_parquet(FEATURES_PATH)
    print('Loaded features:', df.shape)
    df.head()

Model not found at ..\..\models\xgb_patch_model.pkl ; expected at repo root models/xgb_patch_model.pkl
Features file not found at ..\data\processed\features.parquet - skipping loading and visualizations.


In [6]:
# Target distribution
if 'df' in globals() and not df.empty:
    plt.figure(figsize=(6,4))
    ax = sns.countplot(x='blue_side_win', data=df)
    ax.set_title('Target distribution (blue_side_win)')
    plt.show()
    # Summary stats
    display(df['blue_side_win'].describe())
else:
    print('No features loaded; skipping target distribution plot.')

No features loaded; skipping target distribution plot.


In [7]:
# Win rate by patch (top 20 patches by count)
if 'df' in globals() and not df.empty:
    if 'patch_version' in df.columns:
        pv = df.groupby('patch_version').agg(n=('blue_side_win','size'), winrate=('blue_side_win','mean')).reset_index()
        pv_sorted = pv.sort_values('n', ascending=False).head(20)
        plt.figure(figsize=(12,5))
        sns.barplot(y='patch_version', x='winrate', data=pv_sorted, palette='viridis')
        plt.xlabel('Blue side win rate')
        plt.ylabel('Patch (top 20 by count)')
        plt.title('Blue side win rate by patch (top 20)')
        plt.xlim(0,1)
        plt.show()
    else:
        print('No patch_version column found in features.')
else:
    print('No features loaded; skipping patch-level winrate plot.')

No features loaded; skipping patch-level winrate plot.


In [8]:
# Show some example numeric summaries
if 'df' in globals() and not df.empty:
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    print('Numeric columns:', len(num_cols))
    display(df[num_cols].describe().T)
else:
    print('No features loaded; skipping numeric summaries')


No features loaded; skipping numeric summaries


In [9]:
# If a model exists, load and show feature importances
import pickle
import numpy as np
import re
if 'df' in globals() and not df.empty and MODEL_PATH.exists():
    try:
        with open(MODEL_PATH, 'rb') as fh:
            model = pickle.load(fh)
        # try sklearn-style importances first
        if hasattr(model, 'feature_importances_'):
            fi = model.feature_importances_
            feat_names = df.drop(columns=['blue_side_win','patch_version'], errors='ignore').columns.tolist()
            imp = pd.Series(fi, index=feat_names).sort_values(ascending=False).head(30)
        else:
            # try booster get_score
            booster = getattr(model, 'get_booster', lambda: None)()
            if booster is not None:
                scores = booster.get_score(importance_type='gain')
                imp = pd.Series(scores).sort_values(ascending=False).head(30)
            else:
                imp = None
        # Attempt to fetch champion id -> name mapping from Data Dragon
        id_to_name = {}
        try:
            import requests
            ver = requests.get('https://ddragon.leagueoflegends.com/api/versions.json').json()[0]
            cj = requests.get(f'https://ddragon.leagueoflegends.com/cdn/{ver}/data/en_US/champion.json').json()
            id_to_name = {int(v['key']): v['name'] for v in cj['data'].values()}
        except Exception as _e:
            # offline-friendly: keep empty mapping and continue
            print('Could not fetch champion mapping:', _e)
            id_to_name = {}
        def label_champ(col):
            m = re.search(r'_(\d+)$', col)
            if m:
                cid = int(m.group(1))
                name = id_to_name.get(cid)
                if name:
                    return col.rsplit('_',1)[0] + '_' + name
            return col
        if imp is not None and not imp.empty:
            # rename champion id suffixes to champion names when possible
            imp.index = [label_champ(c) for c in imp.index]
            plt.figure(figsize=(8,10))
            sns.barplot(x=imp.values, y=imp.index, palette='magma')
            plt.title('Top feature importances')
            plt.xlabel('Importance')
            plt.show()
        else:
            print('No feature importances available from the model.')
    except Exception as e:
        print('Failed to load model:', e)
else:
    print('Skipping model feature importances (no model file or no features loaded).')

Skipping model feature importances (no model file or no features loaded).
